# HỌC PHẦN: PHƯƠNG PHÁP NGHIÊN CỨU KHOA HỌC TRONG KHTN
## ĐỀ TÀI: NGHIÊN CỨU ỨNG DỤNG MACHINE LEARNING DỰ BÁO BỤI MỊN PM2.5 TẠI ĐÀ NẴNG
### BÁO CÁO THỰC NGHIỆM PILOT (TÁI LẬP THEO CHUẨN FAIR)
---
**Sinh viên thực hiện:** Khoa Công nghệ thông tin - Trường Đại học Sư phạm, ĐH Đà Nẵng  
**Mục tiêu:** Kiểm chứng tính khả thi của mô hình học máy (Random Forest và XGBoost) trong việc dự báo nồng độ PM2.5 kết hợp yếu tố khí tượng tại TP. Đà Nẵng.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Tất cả thư viện khoa học dữ liệu đã sẵn sàng!")

### 1. Nạp và Khảo sát dữ liệu chuỗi thời gian (EDA)

In [ ]:
data_path = "../data/danang_air_quality_pilot_dataset.csv"
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Số lượng bản ghi quan trắc theo giờ: {len(df)}")
df.head()

### 2. Thống kê mô tả và độ không đảm bảo đo mở rộng U95%

In [ ]:
stats = df[['PM25', 'temp', 'humidity', 'wind_speed', 'pressure', 'rain']].describe().T
stats['SE'] = stats['std'] / np.sqrt(stats['count'])
stats['U95'] = 1.96 * stats['SE']
stats[['mean', 'std', 'min', '50%', 'max', 'U95']].round(2)

### 3. Phân tích ma trận tương quan giữa PM2.5 và biến khí tượng ven biển

In [ ]:
corr = df[['PM25', 'temp', 'humidity', 'wind_speed', 'pressure', 'rain']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".3f", vmin=-1, vmax=1)
plt.title("MA TRẬN TƯƠNG QUAN PEARSON GIỮA PM2.5 VÀ KHÍ TƯỢNG ĐÀ NẴNG")
plt.show()

### 4. Kỹ nghệ đặc trưng (Lag Features, Rolling Window, Diurnal)

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)

df['PM25_lag1'] = df['PM25'].shift(1)
df['PM25_lag2'] = df['PM25'].shift(2)
df['PM25_lag24'] = df['PM25'].shift(24)
df['PM25_roll3'] = df['PM25'].rolling(window=3).mean()
df['PM25_roll24'] = df['PM25'].rolling(window=24).mean()

df_clean = df.dropna().reset_index(drop=True)
print(f"Kích thước sau khi làm sạch: {df_clean.shape}")

### 5. Huấn luyện mô hình và Đánh giá sai số (TimeSeriesSplit)

In [ ]:
features = ['PM25_lag1', 'PM25_lag2', 'PM25_lag24', 'PM25_roll3', 'PM25_roll24', 'temp', 'humidity', 'wind_speed', 'pressure', 'rain', 'hour_sin', 'hour_cos']
X = df_clean[features]
y = df_clean['PM25']

split_idx = int(len(df_clean) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Baseline Model
base_preds = X_test['PM25_lag1'].values

# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Gradient Boosting / XGBoost
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)

results = pd.DataFrame({
    'Model': ['Baseline (Naive)', 'Random Forest', 'Gradient Boosting / XGBoost'],
    'MAE': [mean_absolute_error(y_test, base_preds), mean_absolute_error(y_test, rf_preds), mean_absolute_error(y_test, gb_preds)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, base_preds)), np.sqrt(mean_squared_error(y_test, rf_preds)), np.sqrt(mean_squared_error(y_test, gb_preds))],
    'R2_Score': [r2_score(y_test, base_preds), r2_score(y_test, rf_preds), r2_score(y_test, gb_preds)]
})
results.round(3)

### 6. Trực quan hóa kết quả dự báo và Quy chuẩn QCVN 05:2023/BTNMT

In [ ]:
plt.figure(figsize=(14, 5))
test_timeline = df_clean['timestamp'].iloc[split_idx:]
plt.plot(test_timeline, y_test.values, label='PM2.5 Thực tế', color='#1f77b4', linewidth=2)
plt.plot(test_timeline, gb_preds, label=f'Dự báo (R² = {r2_score(y_test, gb_preds):.3f})', color='#d62728', linestyle='--', linewidth=1.8)
plt.axhline(y=25, color='orange', linestyle=':', label='QCVN 05:2023 24h (25 ug/m3)')
plt.axhline(y=50, color='red', linestyle=':', label='QCVN 05:2023 1h (50 ug/m3)')
plt.title('SO SÁNH KẾT QUẢ DỰ BÁO CỦA MÔ HÌNH VỚI DỮ LIỆU THỰC TẾ TẠI ĐÀ NẴNG')
plt.xlabel('Thời gian quan trắc')
plt.ylabel('Nồng độ PM2.5 (ug/m3)')
plt.legend()
plt.show()